# 01 · The `collections` Module

**Module 12 – Standard Library Essentials** · Estimated time: 40 min

> Expands 📗 *Collections Module* (Counter, defaultdict, namedtuple) with `deque`, `OrderedDict`, `ChainMap` and when to use each.

### 🎯 What you will learn
- `Counter`: counting things
- `defaultdict`: dictionaries with automatic defaults
- `namedtuple`: tuples with field names (see also Module 06)
- `deque`: fast queues and stacks
- `OrderedDict`: when (and when not) to use it today
- `ChainMap`: layered lookups

## 1. `Counter` (📗)

A `dict` subclass that counts hashable objects.

In [1]:
from collections import Counter

lst = [1, 2, 2, 2, 2, 3, 3, 3, 1, 2, 1, 12, 3, 2, 32, 1, 21, 1, 223, 1]
print(Counter(lst))
print(Counter("aabsbsbsbhshhbbsbs"))

s = "How many times does each word show up in this sentence word times each each word"
words = s.split()
c = Counter(words)
print(c)
print(c.most_common(2))

Counter({1: 6, 2: 6, 3: 4, 12: 1, 32: 1, 21: 1, 223: 1})
Counter({'b': 7, 's': 6, 'h': 3, 'a': 2})
Counter({'each': 3, 'word': 3, 'times': 2, 'How': 1, 'many': 1, 'does': 1, 'show': 1, 'up': 1, 'in': 1, 'this': 1, 'sentence': 1})
[('each', 3), ('word', 3)]


### Common patterns with `Counter` (📗)

In [2]:
c = Counter("mississippi")
print(sum(c.values()))           # total of all counts
print(list(c))                   # unique elements
print(set(c), dict(c))
print(c.items())
print(c.most_common()[:-3:-1])   # 2 least common elements
c += Counter()                   # remove zero and negative counts
c["z"] += 1                      # missing keys default to 0, no KeyError
print(c["q"], c)

11
['m', 'i', 's', 'p']
{'p', 'i', 's', 'm'} {'m': 1, 'i': 4, 's': 4, 'p': 2}
dict_items([('m', 1), ('i', 4), ('s', 4), ('p', 2)])
[('m', 1), ('p', 2)]
0 Counter({'i': 4, 's': 4, 'p': 2, 'm': 1, 'z': 1})


In [3]:
# Counter arithmetic
a = Counter(apples=3, oranges=1)
b = Counter(apples=1, pears=2)
print(a + b, a - b, a & b, a | b, sep="\n")
print(a.total())                 # 3.10+: sum of counts

Counter({'apples': 4, 'pears': 2, 'oranges': 1})
Counter({'apples': 2, 'oranges': 1})
Counter({'apples': 1})
Counter({'apples': 3, 'pears': 2, 'oranges': 1})
4


## 2. `defaultdict` (📗)

Like a dict, but a missing key is automatically created with a default value from a **factory function**, so you never get a `KeyError`.

In [4]:
from collections import defaultdict

d = {}
try:
    d["one"]
except KeyError as e:
    print("KeyError:", e)

d = defaultdict(object)
d["one"]
print(list(d))

d = defaultdict(lambda: 0)
d["one"]
print(d["one"])

KeyError: 'one'
['one']
0


In [5]:
# The most common uses: grouping and counting
words = ["apple", "avocado", "banana", "blueberry", "cherry", "apricot"]
by_letter = defaultdict(list)
for w in words:
    by_letter[w[0]].append(w)          # no need to check whether the key exists
print(dict(by_letter))

counts = defaultdict(int)              # int() → 0
for w in words:
    counts[len(w)] += 1
print(dict(counts))

graph = defaultdict(set)               # adjacency list
for a, b in [("A", "B"), ("A", "C"), ("B", "C")]:
    graph[a].add(b)
    graph[b].add(a)
print(dict(graph))

{'a': ['apple', 'avocado', 'apricot'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}
{5: 1, 7: 2, 6: 2, 9: 1}
{'A': {'B', 'C'}, 'B': {'A', 'C'}, 'C': {'B', 'A'}}


## 3. `namedtuple` (📗)

In [6]:
from collections import namedtuple

t = (12, 13, 14)
print(t[0])

Dog = namedtuple("Dog", ["age", "breed", "name"])
sam = Dog(age=2, breed="Lab", name="Sammy")
frank = Dog(age=2, breed="Shepard", name="Frankie")
print(sam)
print(sam.age, sam.breed, sam[2])
print(frank._asdict())

12
Dog(age=2, breed='Lab', name='Sammy')
2 Lab Sammy
{'age': 2, 'breed': 'Shepard', 'name': 'Frankie'}


(See Module 06 · Dataclasses for `typing.NamedTuple` and `@dataclass`, the modern alternatives.)

## 4. `deque`: double-ended queue

`list.pop(0)` and `list.insert(0, x)` are **O(n)**. `deque` does both ends in **O(1)**.

In [7]:
from collections import deque

q = deque(["a", "b", "c"])
q.append("d")          # right
q.appendleft("z")      # left
print(q)
print(q.pop(), q.popleft(), q)
q.rotate(1)
print(q)

recent = deque(maxlen=3)               # keeps only the last N items
for page in ["home", "about", "blog", "contact", "shop"]:
    recent.append(page)
print("Last 3 pages:", list(recent))

deque(['z', 'a', 'b', 'c', 'd'])
d z deque(['a', 'b', 'c'])
deque(['c', 'a', 'b'])
Last 3 pages: ['blog', 'contact', 'shop']


In [8]:
import timeit
print("list.pop(0):    ", round(timeit.timeit("l.pop(0); l.append(0)", "l=list(range(100_000))", number=5_000), 4))
print("deque.popleft():", round(timeit.timeit("d.popleft(); d.append(0)",
                                              "from collections import deque; d=deque(range(100_000))", number=5_000), 4))

list.pop(0):     0.073
deque.popleft(): 0.0001


## 5. `OrderedDict`

> ⚠️ **Mostly historical:** before Python 3.7, normal dicts didn't keep insertion order, so `OrderedDict` was essential. Today regular dicts preserve order. `OrderedDict` is still useful for:
> - `move_to_end()` and `popitem(last=False)` (e.g. building an **LRU cache**)
> - order-sensitive equality comparisons

In [9]:
from collections import OrderedDict

print(OrderedDict(a=1, b=2) == OrderedDict(b=2, a=1))   # False: order matters
print(dict(a=1, b=2) == dict(b=2, a=1))                 # True

class LRUCache:
    def __init__(self, capacity):
        self.data = OrderedDict()
        self.capacity = capacity
    def get(self, key):
        if key not in self.data:
            return None
        self.data.move_to_end(key)             # mark as recently used
        return self.data[key]
    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)      # evict the least recently used

cache = LRUCache(2)
cache.put("a", 1); cache.put("b", 2); cache.get("a"); cache.put("c", 3)
print(list(cache.data))                        # 'b' was evicted

False
True
['a', 'c']


## 6. `ChainMap`: search several dicts as one

In [10]:
from collections import ChainMap

defaults = {"theme": "light", "lang": "en", "debug": False}
env_config = {"debug": True}
cli_args = {"lang": "hi"}
config = ChainMap(cli_args, env_config, defaults)   # first match wins
print(config["lang"], config["debug"], config["theme"])
print(dict(config))

hi True light
{'theme': 'light', 'lang': 'hi', 'debug': True}


## 📝 Summary

| Class | Use it for |
|---|---|
| `Counter` | counting, frequency tables, multiset maths |
| `defaultdict` | grouping, counting, nested structures without `KeyError` |
| `namedtuple` | lightweight immutable records |
| `deque` | queues, sliding windows, "last N items" |
| `OrderedDict` | LRU caches, order-sensitive equality |
| `ChainMap` | layered configuration |

## 🏋️ Practice
1. Find the 3 most common words in a paragraph (ignore case and punctuation).
2. Group a list of names by their length using `defaultdict(list)`.
3. Keep a rolling window of the last 5 temperature readings and print the average after each new reading.
4. Check whether two strings are anagrams using `Counter`.

<details><summary>Show solutions</summary>

```python
from collections import Counter, defaultdict, deque
import string

text = "The cat and the hat. The cat sat!"
words = text.lower().translate(str.maketrans("", "", string.punctuation)).split()
print(Counter(words).most_common(3))

groups = defaultdict(list)
for n in ["Asha", "Ravi", "Meena", "Jo"]:
    groups[len(n)].append(n)

window = deque(maxlen=5)
for t in [21, 22, 24, 23, 25, 27, 26]:
    window.append(t)
    print(round(sum(window) / len(window), 2))

print(Counter("listen") == Counter("silent"))
```
</details>